# json库
## 文件的读取与打印
* 读取：```js.load()```
    *
* 打印：```js.dump()```
* ```with open("vectorData.json",'r',encoding='utf-8')```
* 这句中的r/w是读取/写入控制符
    * with关键字可以让python自动关闭文件
## 字符串的转换
* python->json ```js.dumps()```
* json->python ```js.loads()```

In [118]:
import json
from asyncio.windows_events import NULL

import numpy as np
import pandas as pd

# 读取.json文件
with open("vectorData.json",'r',encoding='utf-8') as f:
    data = f.read()
    print(data)


[
    {
        "group_name": "2d_task_1",
        "vectors": [[1,3],[1,2],[2,4],[3,1],[4,3],[5,5],[6,2],[7,7],[8,6],[9,8],[10,9]],
        "ori_axis": [[1,0],[0,1]],
        "tasks": [
            { "type": "axis_angle" },
            { "type": "change_axis", "obj_axis": [[2,1],[1,2]] },
            { "type": "area" },
            { "type": "axis_projection" },
            { "type": "axis_angle"}
        ]
    },
    {
        "group_name": "2d_task_2",
        "vectors": [[1,1],[2,0],[3,5],[4,2],[5,7],[6,4],[7,9],[8,6],[9,1],[10,8],[11,3],[12,10]],
        "ori_axis": [[1,1],[1,-1]],
        "tasks": [
            { "type": "area" },
            { "type": "axis_projection" },
            { "type": "change_axis", "obj_axis": [[3,2],[2,-3]] },
            { "type": "axis_projection"},
            { "type": "change_axis", "obj_axis": [[1,0],[0,1]] },
            { "type": "area"},
            { "type": "axis_angle" }
        ]
    },
    {
        "group_name": "2d_task_3",
        "vec

### 接下来准备正式接收数据

In [119]:
    data = json.loads(data)

    # 摊平数据
    vector_data = pd.json_normalize(
        data,
        # 定义要展开的层级
        record_path=['tasks'],
        # 定义要保留的外层字段
        meta = ['group_name', 'vectors', 'ori_axis'],
        errors='ignore',
    )
    # print(vector_data.info())
    # print(vector_data['type'].head())

    # 由于task中的各种东西过于混乱，我仅仅保留了目标坐标轴向量
    vector_data_clean = vector_data.dropna()
    vector_data_clean = vector_data_clean.drop('type',axis=1).copy()
    print(vector_data_clean.info())
    print(vector_data_clean.head())

    import numpy

    # 下面定义进行四种运算的类
    class AXCaculator:
        def __init__(self, ori_axis, obj_axis, vectors):
            # 将传入的嵌套列表转化为 numpy 数组，方便进行矩阵和向量运算
            # 此时vector被转换为大矩阵，ori/obj向量被转换为了
            # 假设 axis 矩阵的每一行是一个基向量
            self.ori_axis = np.array(ori_axis, dtype=float)
            self.obj_axis = np.array(obj_axis, dtype=float)
            self.vectors = np.array(vectors, dtype=float)

            # 获取维度确定初始的坐标轴向量维度 n
            self.n_dim = self.ori_axis.shape[0]

        # 坐标系转移（接受dataFrame的ori_axis, obj_axis, vectors、在维度为n维的情况下仍成立）
            # 转换公式：数量积/轴向量的模的平方(非正交的基不能使用该方法)
            # 转换： 矩阵的逆
        # 将vector转换为标准坐标系下的vector
        def to_standard(self):
            # 用矩阵乘法
            standard_vector = self.ori_axis @ self.vectors
            # 返回标准坐标系下的vectors
            return standard_vector

        # 检查新的坐标轴向量是否合法（所有的向量需要线性无关）
        def check_validity(self):
            if self.obj_axis.det() == 0:
                print("")
                return False
            return True
        # 再将其转换为新坐标系下的向量
        def to_obj
        # 在新坐标系下完成以下内容
        # 坐标系投影
            # 相应新坐标与轴向量的模相乘
        # 坐标系夹角
            # 求
        # 坐标系面积





<class 'pandas.core.frame.DataFrame'>
Index: 21 entries, 1 to 85
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   obj_axis    21 non-null     object
 1   group_name  21 non-null     object
 2   vectors     21 non-null     object
 3   ori_axis    21 non-null     object
dtypes: object(4)
memory usage: 840.0+ bytes
None
             obj_axis group_name  \
1    [[2, 1], [1, 2]]  2d_task_1   
7   [[3, 2], [2, -3]]  2d_task_2   
9    [[1, 0], [0, 1]]  2d_task_2   
13   [[1, 1], [1, 0]]  2d_task_3   
16   [[1, 0], [0, 1]]  2d_task_3   

                                              vectors           ori_axis  
1   [[1, 3], [1, 2], [2, 4], [3, 1], [4, 3], [5, 5...   [[1, 0], [0, 1]]  
7   [[1, 1], [2, 0], [3, 5], [4, 2], [5, 7], [6, 4...  [[1, 1], [1, -1]]  
9   [[1, 1], [2, 0], [3, 5], [4, 2], [5, 7], [6, 4...  [[1, 1], [1, -1]]  
13  [[0, 5], [1, 4], [2, 3], [3, 2], [4, 1], [5, 0...   [[2, 3], [3, 2]]  
16  [[0, 5], [1, 4], 